# Modeling — Medallion, MVs, Streaming Tables, Lakeflow Spark Declarative Pipelines

## Reference domain

We extend the Fintech bank's silver layer into a **gold layer** that BI and ML consume directly. Two concrete gold artifacts drive the examples:

- **`gold.customer_360`** — one row per customer with rollups across cards / loans / accounts / payments. Refreshed nightly. A **materialized view** is the right answer.
- **`gold.daily_card_volume`** — running daily-grain card volume by merchant category. Updates continuously as new transactions arrive. A **streaming table** is the right answer.

## Medallion architecture — what belongs in each layer

Three layers, each with a single job. The exam tests recognising which layer a description fits into.

```text
  ┌─────────────────────────────────────────────────────────────┐
  │  Bronze — raw, append-only, replayable                       │
  │   ▪ One table per source                                     │
  │   ▪ Schema mirrors source (nested OK)                        │
  │   ▪ Strings + nulls allowed; quality is NOT enforced here    │
  │   ▪ This is your audit copy                                  │
  ├─────────────────────────────────────────────────────────────┤
  │  Silver — cleaned, conformed, deduplicated                   │
  │   ▪ Same grain as bronze (one row per event)                 │
  │   ▪ Types pinned; nulls in required columns dropped          │
  │   ▪ Conformed FKs (customer_id is STRING everywhere)         │
  │   ▪ Quality rules ENFORCED (expectations / CHECK)            │
  ├─────────────────────────────────────────────────────────────┤
  │  Gold — business-ready, aggregated, served                   │
  │   ▪ Star schema or wide rollup; ready for BI / ML            │
  │   ▪ Often pre-aggregated (daily, customer-level)             │
  │   ▪ One or more rows per business entity, NOT per event      │
  │   ▪ Permissions tightest — analysts / consumers join here    │
  └─────────────────────────────────────────────────────────────┘
```

**Important rule the exam tests:** you can have *multiple* gold tables driven by the *same* silver tables. The bank's `gold.customer_360`, `gold.daily_card_volume`, and `gold.fraud_features` all read from the same `silver.card_transactions`. Don't try to make one giant gold table — make many narrow ones, each shaped to its consumer.

## The five object types you can build in gold

The exam expects you to pick the right one for each scenario. Memorise these:

| Object | What it is | Storage? | Latency | Right when… |
|---|---|---|---|---|
| **Table** | Plain Delta table you write to with batch jobs | ✅ | Cadence-bound (depends on job) | You need full control over how & when it's written |
| **View** | Logical query, evaluated on read | ❌ | None — recomputes every read | Light wrapping, security barriers, hide PII columns |
| **Materialized View (MV)** | Precomputed query result, stored, *refreshed automatically* | ✅ | Refresh-bound (declarative cadence) | Heavy aggregation read often — `customer_360` |
| **Streaming Table** | Append-only table fed by a Structured Streaming query | ✅ | Near-real-time | Continuous CDC / fact ingestion |
| **MV / Streaming Table on Lakeflow Spark Declarative Pipelines** | Same shapes, *defined declaratively* with expectations + lineage | ✅ | Same as above | You want the pipeline framework's quality + lineage built in |

**The two new objects vs. classic Delta:**

- **Materialized View** — Databricks computes and stores the query result. When base tables change, the MV is refreshed (incrementally where possible) on its schedule. You query it like any table — fast and consistent. *This is the right answer when a question describes "heavy aggregation that BI hits a lot."*
- **Streaming Table** — append-only table where rows arrive continuously from a Structured Streaming source. Same checkpointing semantics as raw streaming. *This is the right answer when a question describes "records flow in continuously."*

## Plain views — security barriers, hiding columns, light wrapping

A view stores no data — it's a saved SELECT statement, evaluated every time it's read. Three idiomatic uses:

**Security barrier.** Hide PII columns from a downstream catalog:

```sql
 CREATE VIEW fintech_dev.gold.customers_public AS
 SELECT customer_id, city, country, created_at
 FROM   fintech_dev.silver.customers;
```

Then `GRANT SELECT ON VIEW customers_public TO analyst_group`. The analysts can't see emails / phones / DOBs because the view doesn't expose them.

**Light wrapping.** Encode a frequently-used filter or join once, share it everywhere:

```sql
 CREATE VIEW fintech_dev.gold.recent_card_txns AS
 SELECT * FROM fintech_dev.silver.card_transactions
 WHERE transaction_at >= current_date() - INTERVAL 7 DAYS;
```

**Temporary views** in a notebook session:

```python
 df.createOrReplaceTempView("todays_txns")        # session-scoped
 df.createOrReplaceGlobalTempView("todays_txns")  # cross-session, prefixed global_temp.
```

**The trade-off the exam tests:** a view recomputes on every read. For a heavy aggregation hit by a thousand dashboard refreshes a day, a **materialized view** is the right choice; a plain view would re-run the aggregation every time.

## Materialized views — precomputed, auto-refreshed, queried like a table

A **materialized view (MV)** is a query result that Databricks computes once, stores as Delta, and **refreshes** on a declared cadence (or whenever you call `REFRESH MATERIALIZED VIEW`).

```sql
 CREATE OR REPLACE MATERIALIZED VIEW fintech_dev.gold.customer_360
 SCHEDULE EVERY 1 HOUR
 AS
 SELECT c.customer_id,
        c.full_name,
        c.city,
        COUNT(t.transaction_id)                                  AS txn_count_30d,
        SUM(t.amount)                                            AS total_spend_30d,
        APPROX_COUNT_DISTINCT(t.merchant_name)                   AS distinct_merchants_30d
 FROM   fintech_dev.silver.customers c
 LEFT JOIN fintech_dev.silver.card_transactions t
    ON   c.customer_id = t.customer_id
   AND   t.transaction_at >= current_date() - INTERVAL 30 DAYS
 GROUP BY c.customer_id, c.full_name, c.city;
```

What you get:

- **Incremental refresh** — when the base tables change, only the affected rows recompute (when the query supports it).
- **Consistent reads** — like a table, not subject to base-table mid-write inconsistency.
- **Cheap for BI** — analysts hit a precomputed result, not a 50-billion-row aggregation.
- **Defined declaratively** — schedule lives with the object, not in a separate Lakeflow Jobs definition.

MVs are the **right answer** whenever a question describes "a heavy aggregation that BI reads frequently, refresh on a cadence."

## Streaming tables — append-only, continuous, declarative

A **streaming table** is an append-only Delta table fed by a Structured Streaming query, declared as a first-class UC object instead of as imperative `readStream / writeStream` code.

```sql
 CREATE OR REFRESH STREAMING TABLE fintech_dev.bronze.card_transactions_raw
 AS
 SELECT * FROM STREAM read_files(
   '/Volumes/fintech_dev/bronze/landing_zone/cards/',
   format => 'json',
   schemaEvolutionMode => 'addNewColumns'
 );
```

The `read_files(... STREAM ...)` reader is the SQL surface of Auto Loader. You get the same checkpointing and exactly-once semantics — but the source is just a SQL function call.

**Two pipeline modes that drive cost and latency:**

- **Triggered** — the pipeline runs to completion against whatever data is available, then shuts down. The streaming-tables-on-a-schedule pattern. Lower cost.
- **Continuous** — the pipeline runs forever, processing data as it arrives. Lower latency, higher cost.

Streaming tables are the **right answer** whenever a question describes "records arrive continuously and the table should update continuously."

# Lakeflow Spark Declarative Pipelines

Plain tables, views, MVs, and streaming tables are useful on their own. **Lakeflow Spark Declarative Pipelines** — formerly known as **Delta Live Tables (DLT)** — wrap them in a framework that gives you:

- **Declarative dataset definitions** — you describe *what* each dataset is; the framework figures out the DAG and order.
- **Expectations** — declarative data quality rules attached to each dataset.
- **Automatic lineage** — captured at definition time, visible in Unity Catalog.
- **CDC + SCD via `APPLY CHANGES INTO`** — declarative upserts without writing MERGE plumbing.
- **Managed compute and retry** — Databricks handles cluster lifecycle and per-dataset retries.

**The rename watch.** *Delta Live Tables / DLT* is the legacy name; *Lakeflow Spark Declarative Pipelines* is current. The exam can use either. The decorators are still `@dlt.table` / `@dlt.view` for backwards compatibility, and the SQL keywords are still `STREAMING TABLE` and `MATERIALIZED VIEW`.

## Defining datasets — Python and SQL

**Python (decorator-based):**

```python
 import dlt
 from pyspark.sql.functions import col

 @dlt.table(comment="Raw card transactions landed from SFTP")
 def bronze_card_transactions():
     return (spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .load("/Volumes/fintech_dev/bronze/landing_zone/cards/"))

 @dlt.table(comment="Cleaned card transactions")
 def silver_card_transactions():
     return (dlt.read_stream("bronze_card_transactions")
         .where(col("customer_id").isNotNull())
         .withColumn("amount", col("amount").cast("decimal(18,2)")))

 @dlt.table(comment="Daily card volume by merchant category")
 def gold_daily_card_volume():
     return (dlt.read("silver_card_transactions")
         .groupBy("merchant_category", "transaction_date")
         .agg({"amount": "sum", "transaction_id": "count"}))
```

**SQL (declarative, same DAG):**

```sql
 CREATE OR REFRESH STREAMING TABLE bronze_card_transactions
   COMMENT 'Raw card transactions landed from SFTP'
 AS SELECT * FROM STREAM read_files('/Volumes/fintech_dev/bronze/landing_zone/cards/', format => 'json');

 CREATE OR REFRESH STREAMING TABLE silver_card_transactions
   COMMENT 'Cleaned card transactions'
 AS SELECT * FROM STREAM(bronze_card_transactions)
    WHERE customer_id IS NOT NULL;

 CREATE OR REFRESH MATERIALIZED VIEW gold_daily_card_volume
   COMMENT 'Daily card volume by merchant category'
 AS SELECT merchant_category, DATE(transaction_at) AS transaction_date,
           SUM(amount) AS total_amount, COUNT(*) AS txn_count
    FROM silver_card_transactions
    GROUP BY merchant_category, DATE(transaction_at);
```

**Two reader functions to memorise:**

- **`dlt.read("name")`** — read another dataset in the same pipeline as a **batch** snapshot.
- **`dlt.read_stream("name")`** — read another dataset in the same pipeline as a **stream**.

## Expectations — declarative data quality

Expectations are predicates you attach to a dataset. When a row arrives that violates the predicate, the pipeline does one of three things based on which keyword you used.

| Keyword | If a row violates the predicate |
|---|---|
| `expect` | Row is **kept**; violation is **logged** to the event log |
| `expect or drop` | Row is **dropped**; violation is logged |
| `expect or fail` | Pipeline **fails** on first violation |

**Python:**

```python
 @dlt.table
 @dlt.expect_all({
     "valid_customer": "customer_id IS NOT NULL",
     "positive_amount": "amount > 0"
 })
 @dlt.expect_or_drop("reasonable_amount", "amount < 1000000")
 @dlt.expect_or_fail("valid_currency", "currency IN ('INR', 'USD', 'EUR')")
 def silver_card_transactions():
     return dlt.read_stream("bronze_card_transactions")
```

**SQL:**

```sql
 CREATE OR REFRESH STREAMING TABLE silver_card_transactions (
   CONSTRAINT valid_customer    EXPECT (customer_id IS NOT NULL),
   CONSTRAINT positive_amount   EXPECT (amount > 0),
   CONSTRAINT reasonable_amount EXPECT (amount < 1000000) ON VIOLATION DROP ROW,
   CONSTRAINT valid_currency    EXPECT (currency IN ('INR','USD','EUR')) ON VIOLATION FAIL UPDATE
 )
 AS SELECT * FROM STREAM(bronze_card_transactions);
```

**The three patterns the exam contrasts:**

- *Soft data quality — track but don't block* → `expect`.
- *Hard data quality — drop bad rows but keep the pipeline running* → `expect or drop`.
- *Critical data quality — a violation means stop and page a human* → `expect or fail`.

## `APPLY CHANGES INTO` — declarative CDC and SCD

Hand-writing `MERGE INTO` for change data capture works (notebook 02) but it's verbose and easy to get wrong. **`APPLY CHANGES INTO`** is the Lakeflow Spark Declarative Pipelines equivalent — declarative CDC with SCD type 1 or type 2 baked in.

```sql
 CREATE OR REFRESH STREAMING TABLE silver_customers;

 APPLY CHANGES INTO live.silver_customers
 FROM   STREAM(live.bronze_customers_cdc)
 KEYS   (customer_id)
 APPLY AS DELETE     WHEN _change_type = 'delete'
 SEQUENCE BY _commit_version
 COLUMNS    * EXCEPT (_change_type, _commit_version)
 STORED AS SCD TYPE 1;     -- or SCD TYPE 2 for history
```

**Five clauses worth memorising:**

- **`KEYS (...)`** — the natural key for the upsert.
- **`SEQUENCE BY ...`** — the ordering column that decides "latest wins" for late-arriving changes.
- **`APPLY AS DELETE WHEN ...`** — propagate hard deletes from the source.
- **`STORED AS SCD TYPE 1`** — overwrite in place, no history kept.
- **`STORED AS SCD TYPE 2`** — keep history with start/end timestamps and an is-current flag, all maintained for you.

**Why the exam loves it:** the SCD type 2 hand-rolled MERGE pattern is a multi-statement, race-prone affair. `APPLY CHANGES INTO ... STORED AS SCD TYPE 2` is one declarative block that does it all correctly.

## Pipeline modes, settings, and the development workflow

A Lakeflow Spark Declarative Pipeline is configured once and re-run many times. Three knobs the exam tests:

**Triggered vs. Continuous.**

- *Triggered* — the pipeline runs once against all available data and stops. Best for scheduled batch.
- *Continuous* — the pipeline runs forever, processing new data as it arrives. Best for real-time streaming.

**Development vs. Production.**

- *Development mode* — cluster stays alive between runs (fast iteration), errors don't kill the cluster.
- *Production mode* — cluster terminates between runs (cheaper), errors fail the run.

**Channel.**

- *Current* — the latest GA runtime.
- *Preview* — early-access features.

**Target catalog and schema** — every pipeline lands its datasets into a UC catalog + schema you choose. Without that, datasets are visible only to the pipeline.

## CHECK constraints on plain Delta — quality without a pipeline

Not every table needs a full pipeline framework. Plain Delta tables support **CHECK constraints** — predicates enforced on every write.

```sql
 ALTER TABLE fintech_dev.silver.card_transactions
   ADD CONSTRAINT positive_amount CHECK (amount > 0);

 ALTER TABLE fintech_dev.silver.card_transactions
   ADD CONSTRAINT valid_currency CHECK (currency IN ('INR','USD','EUR'));
```

**The difference from expectations:**

- CHECK constraints **fail the write** on violation. There's no drop-and-log option.
- They're table-level, not pipeline-level. They apply to *every* write, no matter who's writing.
- They're cheaper to set up — one ALTER TABLE — but less expressive.

**When the exam contrasts them:** if the question says "reject bad rows at write time on a plain Delta table," the answer is CHECK constraints. If the question says "track violations / drop bad rows / fail the pipeline based on severity," the answer is **expectations**.

## Modeling gold — the patterns BI and ML want

Two patterns cover most gold tables.

**Star schema** — facts surrounded by conformed dimensions. The bank's analytical reports run on:

- `gold.fact_card_transactions` (one row per transaction, FKs to dimensions)
- `gold.dim_customer`, `gold.dim_merchant`, `gold.dim_date`

Dimensions usually slowly-change. Customer's city changes? SCD type 2 captures the change while preserving history — `APPLY CHANGES INTO ... STORED AS SCD TYPE 2` is the right tool.

**Wide rollup / OBT (one big table)** — one row per business entity, every interesting metric inline.

- `gold.customer_360` — customer_id + 30+ rollup columns. Easier for analysts to query (no joins), but expensive to refresh.

Choose by *consumer*. Power BI / Tableau dashboards: star schema. Ad-hoc analytics teams / ML feature engineering: OBT.

**The bank's gold layout that drives this notebook's examples:**

```text
  gold.customer_360            — materialized view, refresh hourly
  gold.daily_card_volume       — streaming table, continuous trigger
  gold.dim_customer            — APPLY CHANGES INTO ... SCD TYPE 2
  gold.fact_card_transactions  — APPLY CHANGES INTO ... SCD TYPE 1
```

## The decision sheet — picking the right gold object

| Scenario | Pick |
|---|---|
| Heavy aggregation that BI dashboards refresh many times an hour | **Materialized view** |
| Continuous fact ingestion (card transactions stream in, gold table should reflect them) | **Streaming table** |
| Light wrapper / hide PII columns / encode a common filter | **View** |
| Full declarative pipeline with quality gates, lineage, CDC | **Lakeflow Spark Declarative Pipelines** |
| Need SCD type 2 history on a dimension fed by CDC | **`APPLY CHANGES INTO ... STORED AS SCD TYPE 2`** |
| Need to drop rows that fail a quality rule but keep the pipeline going | **`expect or drop`** |
| Need to stop the pipeline on a critical violation | **`expect or fail`** |
| Reject bad rows at write time on a plain Delta table | **`CHECK` constraint** |
| Custom batch logic you fully control end to end | **Plain Delta table + Lakeflow Jobs task** |

## Hands-on — gold objects and quality on the Fintech bank

The cells below assume `fintech_dev.silver.card_transactions` from notebook 04 exists. They:

1. Create a plain view as a security barrier.
2. Create a materialized view for `customer_360`.
3. Create a streaming table that reads silver as a stream and aggregates daily volume.
4. Add CHECK constraints to silver.
5. Sketch a full Lakeflow Spark Declarative Pipeline Python module — to be deployed via Lakeflow Jobs in notebook 06.

The full Lakeflow Spark Declarative Pipeline cell is a code recipe — to run it, save it as `pipeline.py` and create a pipeline in the workspace UI.

In [ ]:
spark.sql("USE CATALOG fintech_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [ ]:
# A plain view that hides PII — analysts who only have GRANT SELECT on this view
# cannot see emails, phones, or DOBs from the silver customers table.
spark.sql("""
    CREATE OR REPLACE VIEW gold.customers_public AS
    SELECT customer_id, city, country, created_at
    FROM   silver.customers
""")
spark.sql("DESCRIBE TABLE gold.customers_public").show(truncate=False)

In [ ]:
# customer_360 as a materialized view — refresh hourly, BI hits a precomputed result.
# (On a SQL warehouse / DBR with MV support; otherwise, the equivalent is a CTAS rebuilt nightly.)
spark.sql("""
    CREATE OR REPLACE MATERIALIZED VIEW gold.customer_360
    SCHEDULE EVERY 1 HOUR
    AS
    SELECT c.customer_id,
           c.city,
           COUNT(t.transaction_id)                AS txn_count_30d,
           SUM(t.amount)                          AS total_spend_30d,
           APPROX_COUNT_DISTINCT(t.merchant_name) AS distinct_merchants_30d
    FROM   silver.customers c
    LEFT JOIN silver.card_transactions t
      ON   c.customer_id = t.customer_id
     AND   t.transaction_at >= current_date() - INTERVAL 30 DAYS
    GROUP BY c.customer_id, c.city
""")
spark.sql("SELECT * FROM gold.customer_360 ORDER BY total_spend_30d DESC NULLS LAST LIMIT 5").show(truncate=False)

In [ ]:
# Streaming table for gold.daily_card_volume — silver_card_transactions consumed as a stream,
# rolled up to per-day, per-category volume. Runs continuously when scheduled in a pipeline.
spark.sql("""
    CREATE OR REFRESH STREAMING TABLE gold.daily_card_volume
    AS
    SELECT merchant_category,
           DATE(transaction_at) AS transaction_date,
           SUM(amount)          AS total_amount,
           COUNT(*)             AS txn_count
    FROM   STREAM(silver.card_transactions)
    GROUP BY merchant_category, DATE(transaction_at)
""")

In [ ]:
# CHECK constraints on plain Delta — every write to silver.card_transactions is now validated.
spark.sql("""
    ALTER TABLE silver.card_transactions
      ADD CONSTRAINT positive_amount CHECK (amount > 0)
""")
spark.sql("""
    ALTER TABLE silver.card_transactions
      ADD CONSTRAINT valid_currency CHECK (currency IS NULL OR currency IN ('INR','USD','EUR'))
""")
spark.sql("SHOW TBLPROPERTIES silver.card_transactions").show(truncate=False)

In [ ]:
# Recipe — save this as pipeline.py, register it as a Lakeflow Spark Declarative Pipeline
# in the workspace UI, point its target at fintech_dev.gold, and click Start.
PIPELINE_SOURCE = '''
import dlt
from pyspark.sql.functions import col

# ── Bronze ────────────────────────────────────────────────────────────
@dlt.table(comment="Raw card transactions from SFTP drop")
def bronze_card_transactions():
    return (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/fintech_dev/bronze/landing_zone/_schemas/cards")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .load("/Volumes/fintech_dev/bronze/landing_zone/cards/"))

# ── Silver, with expectations ─────────────────────────────────────────
@dlt.table(comment="Cleaned card transactions")
@dlt.expect_all({"valid_customer": "customer_id IS NOT NULL"})
@dlt.expect_or_drop("reasonable_amount", "amount BETWEEN 0 AND 1000000")
@dlt.expect_or_fail("valid_currency",   "currency IS NULL OR currency IN ('INR','USD','EUR')")
def silver_card_transactions():
    return (dlt.read_stream("bronze_card_transactions")
        .withColumn("amount",         col("amount").cast("decimal(18,2)"))
        .withColumn("is_flagged",     col("is_flagged").cast("boolean")))

# ── Gold — rollup MV ──────────────────────────────────────────────────
@dlt.table(comment="Daily card volume by merchant category")
def gold_daily_card_volume():
    return (dlt.read("silver_card_transactions")
        .groupBy("merchant_category")
        .agg({"amount": "sum", "transaction_id": "count"}))
'''
print(PIPELINE_SOURCE)

In [ ]:
# APPLY CHANGES INTO recipe — declarative SCD type 2 for the customer dimension.
# Save with the pipeline.py above.
SCD2_SOURCE = '''
-- bronze_customers_cdc has columns: customer_id, city, email, _change_type, _commit_version

CREATE OR REFRESH STREAMING TABLE silver_customers_history;

APPLY CHANGES INTO live.silver_customers_history
FROM   STREAM(live.bronze_customers_cdc)
KEYS   (customer_id)
APPLY AS DELETE     WHEN _change_type = \\'delete\\'
SEQUENCE BY _commit_version
COLUMNS    * EXCEPT (_change_type, _commit_version)
STORED AS SCD TYPE 2;
'''
print(SCD2_SOURCE)

## Section 3 (part 2) self-check

**1.** Which gold-layer object is the right answer for a heavy aggregation that BI dashboards read continuously, refreshed on an hourly schedule?

- A. A plain view
- B. A materialized view
- C. A streaming table
- D. A temporary view

**2.** A team needs declarative CDC into a silver customer table, keeping full history of changes with start/end timestamps. Which is the simplest correct approach?

- A. Hand-write a multi-step `MERGE INTO`
- B. `APPLY CHANGES INTO ... STORED AS SCD TYPE 1`
- C. `APPLY CHANGES INTO ... STORED AS SCD TYPE 2`
- D. CHECK constraint

**3.** An expectation should *drop bad rows* but the pipeline must keep running. Which keyword fits?

- A. `expect`
- B. `expect or drop`
- C. `expect or fail`
- D. `CHECK`

**4.** A scenario describes records arriving continuously and the gold table should update continuously with low latency. Which gold object is correct?

- A. Materialized view
- B. View
- C. Streaming table
- D. Plain Delta table refreshed nightly

**5.** Which is the **current** name for *Delta Live Tables*?

- A. Lakeflow Connect
- B. Lakeflow Jobs
- C. Lakeflow Spark Declarative Pipelines
- D. Automation Bundles

<details><summary>Show answers</summary>

1. **B** — MV is precomputed and stored, perfect for repeated BI reads.
2. **C** — `STORED AS SCD TYPE 2` keeps history with start/end timestamps automatically.
3. **B** — `expect or drop` drops violating rows and continues.
4. **C** — streaming tables are append-only and continuous.
5. **C** — DLT was renamed Lakeflow Spark Declarative Pipelines.

</details>